In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/basharanwar/dental-opg/Dental OPG XRAY Dataset/Dental OPG (Classification)/BDC-BDR/101.jpg
/kaggle/input/datasets/basharanwar/dental-opg/Dental OPG XRAY Dataset/Dental OPG (Classification)/BDC-BDR/106.jpg
/kaggle/input/datasets/basharanwar/dental-opg/Dental OPG XRAY Dataset/Dental OPG (Classification)/BDC-BDR/107.jpg
/kaggle/input/datasets/basharanwar/dental-opg/Dental OPG XRAY Dataset/Dental OPG (Classification)/BDC-BDR/109.jpg
/kaggle/input/datasets/basharanwar/dental-opg/Dental OPG XRAY Dataset/Dental OPG (Classification)/BDC-BDR/11.jpg
/kaggle/input/datasets/basharanwar/dental-opg/Dental OPG XRAY Dataset/Dental OPG (Classification)/BDC-BDR/110.jpg
/kaggle/input/datasets/basharanwar/dental-opg/Dental OPG XRAY Dataset/Dental OPG (Classification)/BDC-BDR/114.jpg
/kaggle/input/datasets/basharanwar/dental-opg/Dental OPG XRAY Dataset/Dental OPG (Classification)/BDC-BDR/119.jpg
/kaggle/input/datasets/basharanwar/dental-opg/Dental OPG XRAY Dataset/Dental OPG (Classif

In [2]:
!pip install -q -U ultralytics
import glob
print(glob.glob('/kaggle/input/**/data.yaml', recursive=True))

[]


In [3]:
import os, glob, yaml
found = {}
for r, d, f in os.walk('/kaggle/input'):
    if os.path.basename(r) in ('train', 'valid', 'val', 'test') and 'images' in d and 'labels' in d:
        found.setdefault(os.path.basename(r), r)
print(found)

mx = 0
for p in glob.glob(found['train'] + '/labels/*.txt'):
    for line in open(p):
        if line.strip(): mx = max(mx, int(line.split()[0]))
val_dir = found.get('valid', found.get('val'))
cfg = {'train': found['train'] + '/images', 'val': val_dir + '/images',
       'test': found['test'] + '/images',
       'names': {i: f'class{i}' for i in range(mx + 1)}}
yaml.safe_dump(cfg, open('/kaggle/working/data.yaml', 'w'))
for k in ('train', 'val', 'test'):
    print(k, len(os.listdir(cfg[k])), 'images')
print('classes:', mx + 1)

{'test': '/kaggle/input/datasets/basharanwar/dental-opg/Dental OPG XRAY Dataset/Dental OPG (Object Detection)/Augmented Dataset/test', 'train': '/kaggle/input/datasets/basharanwar/dental-opg/Dental OPG XRAY Dataset/Dental OPG (Object Detection)/Augmented Dataset/train', 'valid': '/kaggle/input/datasets/basharanwar/dental-opg/Dental OPG XRAY Dataset/Dental OPG (Object Detection)/Augmented Dataset/valid'}
train 558 images
val 23 images
test 23 images
classes: 6


In [4]:
!pip install -q -U ultralytics


In [5]:
import os, glob, yaml
from ultralytics import YOLO

base = '/kaggle/input/datasets/basharanwar/dental-opg/Dental OPG XRAY Dataset/Dental OPG (Object Detection)/Augmented Dataset'
cfg = {'train': base + '/train/images', 'val': base + '/valid/images', 'test': base + '/test/images',
       'names': {i: f'class{i}' for i in range(6)}}
yaml.safe_dump(cfg, open('/kaggle/working/data.yaml', 'w'))

res = {}
for name in ['yolov8n.pt', 'yolo11n.pt', 'yolo26n.pt']:
    m = YOLO(name)
    m.train(data='/kaggle/working/data.yaml', epochs=50, patience=10, imgsz=640,
            seed=0, device=0, project='/kaggle/working/runs', name=name[:-3], verbose=False)
    v = m.val(data='/kaggle/working/data.yaml', split='test', plots=True)
    res[name] = dict(P=v.box.mp, R=v.box.mr, map50=v.box.map50, map5095=v.box.map)
    print(name, {k: round(x * 100, 2) for k, x in res[name].items()})

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.188 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, fre

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/50      2.08G      1.842      2.527      1.582        262        640: 100% ━━━━━━━━━━━━ 35/35 9.4it/s 3.7s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 4.1it/s 0.2s
                   all         23        115      0.425      0.294      0.147     0.0677

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      2.08G      1.754      2.293      1.503        354        640: 100% ━━━━━━━━━━━━ 35/35 9.5it/s 3.7s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.8it/s 0.3s
                   all         23        115      0.566      0.206      0.209     0.0915

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50      2.08G      1.716      2.173      1.488        312        640: 100% ━━━━━━━━━━━━ 35/35 9.5it/s 3.7s
                 Class     Images  Instances      Box(P          R

In [6]:
import os, hashlib, numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import datasets, transforms, models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             precision_score, recall_score, multilabel_confusion_matrix)

device = 'cuda'
ROOT = '/kaggle/input/datasets/basharanwar/dental-opg/Dental OPG XRAY Dataset/Dental OPG (Classification)'
use = ['BDC-BDR', 'Caries', 'Fractured Teeth', 'Impacted teeth', 'Infection']

# 1) remove duplicate files across folders -> one multi-label record per unique image
base = datasets.ImageFolder(ROOT)
uniq = {}
for p, y in base.samples:
    with open(p, 'rb') as f: h = hashlib.md5(f.read()).hexdigest()
    uniq.setdefault(h, {'path': p, 'labels': set()})['labels'].add(base.classes[y])
items = list(uniq.values())
paths = [it['path'] for it in items]
Y = np.array([[1.0 if c in it['labels'] else 0.0 for c in use] for it in items], dtype=np.float32)
N = len(paths); print('unique images:', N)

# 2) split by image (162/34/36), choose a seed where val/test have >=2 positives per label
for seed in range(500):
    idx = np.random.RandomState(seed).permutation(N)
    tr, va, te = idx[:162], idx[162:196], idx[196:]
    if Y[va].sum(0).min() >= 2 and Y[te].sum(0).min() >= 2 and Y[tr].sum(0).min() >= 1: break
print('split seed', seed, '| test positives per label:', Y[te].sum(0))

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tf_tr = transforms.Compose([transforms.Resize((256, 512)), transforms.RandomAffine(5, translate=(0.03, 0.03)),
                            transforms.ColorJitter(0.2, 0.2), transforms.ToTensor(), norm])
tf_ev = transforms.Compose([transforms.Resize((256, 512)), transforms.ToTensor(), norm])

class ML(Dataset):
    def __init__(s, ids, tf): s.ids, s.tf = ids, tf
    def __len__(s): return len(s.ids)
    def __getitem__(s, i):
        j = s.ids[i]
        return s.tf(Image.open(paths[j]).convert('RGB')), torch.tensor(Y[j])

dl_tr = DataLoader(ML(tr, tf_tr), batch_size=16, shuffle=True, num_workers=2)
dl_va = DataLoader(ML(va, tf_ev), batch_size=16, num_workers=2)
dl_te = DataLoader(ML(te, tf_ev), batch_size=16, num_workers=2)

def build(arch):
    L = len(use)
    if arch == 'resnet50':
        m = models.resnet50(weights='IMAGENET1K_V2'); m.fc = nn.Linear(m.fc.in_features, L)
    elif arch == 'densenet121':
        m = models.densenet121(weights='IMAGENET1K_V1'); m.classifier = nn.Linear(m.classifier.in_features, L)
    elif arch == 'efficientnet_b0':
        m = models.efficientnet_b0(weights='IMAGENET1K_V1'); m.classifier[1] = nn.Linear(m.classifier[1].in_features, L)
    elif arch == 'inception_v3':
        m = models.inception_v3(weights='IMAGENET1K_V1', aux_logits=True)
        m.aux_logits = False; m.AuxLogits = None; m.fc = nn.Linear(m.fc.in_features, L)
    return m.to(device)

def predict(m, dl):
    m.eval(); P, T = [], []
    with torch.no_grad():
        for x, y in dl:
            P.append(torch.sigmoid(m(x.to(device))).cpu().numpy()); T.append(y.numpy())
    return np.concatenate(P), np.concatenate(T)

pos = Y[tr].mean(0)
pw = torch.tensor((1 - pos) / pos, dtype=torch.float32, device=device)
summary = {}
for arch in ['resnet50', 'densenet121', 'efficientnet_b0', 'inception_v3']:
    torch.manual_seed(0)
    m = build(arch)
    opt = torch.optim.Adam(m.parameters(), lr=1e-4)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pw)
    best, best_state = -1, None
    for ep in range(15):
        m.train()
        for x, y in dl_tr:
            opt.zero_grad(); lossf(m(x.to(device)), y.to(device)).backward(); opt.step()
        P, T = predict(m, dl_va)
        auc = np.mean([roc_auc_score(T[:, k], P[:, k]) for k in range(len(use))])
        if auc > best: best, best_state = auc, {k: v.clone() for k, v in m.state_dict().items()}
    m.load_state_dict(best_state)
    P, T = predict(m, dl_te); pred = (P >= 0.5).astype(int)
    aucs = [roc_auc_score(T[:, k], P[:, k]) for k in range(len(use))]
    aps = [average_precision_score(T[:, k], P[:, k]) for k in range(len(use))]
    cms = multilabel_confusion_matrix(T, pred)   # per label: [[TN, FP], [FN, TP]]
    acc = np.mean([(cm[0, 0] + cm[1, 1]) / cm.sum() for cm in cms])
    summary[arch] = dict(val_auc=best, acc=acc * 100,
        prec=precision_score(T, pred, average='macro', zero_division=0) * 100,
        rec=recall_score(T, pred, average='macro', zero_division=0) * 100,
        f1=f1_score(T, pred, average='macro', zero_division=0) * 100,
        auc=np.mean(aucs), mAP=np.mean(aps))
    print(arch, {k: round(v, 3) for k, v in summary[arch].items()})
    for k, c in enumerate(use): print('   ', c, 'AUC %.2f AP %.2f' % (aucs[k], aps[k]), 'TN/FP/FN/TP', cms[k].ravel())
    fig, ax = plt.subplots(1, len(use), figsize=(3 * len(use), 3))
    for k, c in enumerate(use):
        ax[k].imshow(cms[k], cmap='Blues'); ax[k].set_title(c, fontsize=8)
        for a in range(2):
            for b in range(2): ax[k].text(b, a, cms[k][a, b], ha='center', va='center')
        ax[k].set_xticks([0, 1]); ax[k].set_yticks([0, 1])
        ax[k].set_xlabel('Predicted'); ax[k].set_ylabel('True')
    plt.tight_layout(); plt.savefig(f'/kaggle/working/confusion_{arch}.png', dpi=150); plt.close()
print(summary)

unique images: 232
split seed 1 | test positives per label: [          7          19           4          12           5]
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 214MB/s]


resnet50 {'val_auc': np.float64(0.639), 'acc': np.float64(67.778), 'prec': 26.023, 'rec': 31.378, 'f1': 27.697, 'auc': np.float64(0.592), 'mAP': np.float64(0.415)}
    BDC-BDR AUC 0.83 AP 0.54 TN/FP/FN/TP [24  5  4  3]
    Caries AUC 0.57 AP 0.58 TN/FP/FN/TP [10  7 10  9]
    Fractured Teeth AUC 0.63 AP 0.38 TN/FP/FN/TP [32  0  4  0]
    Impacted teeth AUC 0.56 AP 0.44 TN/FP/FN/TP [10 14  4  8]
    Infection AUC 0.37 AP 0.14 TN/FP/FN/TP [26  5  5  0]
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 146MB/s]


densenet121 {'val_auc': np.float64(0.774), 'acc': np.float64(71.667), 'prec': 24.5, 'rec': 26.717, 'f1': 24.762, 'auc': np.float64(0.565), 'mAP': np.float64(0.367)}
    BDC-BDR AUC 0.72 AP 0.34 TN/FP/FN/TP [25  4  6  1]
    Caries AUC 0.54 AP 0.59 TN/FP/FN/TP [11  6  9 10]
    Fractured Teeth AUC 0.38 AP 0.11 TN/FP/FN/TP [32  0  4  0]
    Impacted teeth AUC 0.68 AP 0.61 TN/FP/FN/TP [12 12  4  8]
    Infection AUC 0.50 AP 0.19 TN/FP/FN/TP [30  1  5  0]
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 150MB/s]


efficientnet_b0 {'val_auc': np.float64(0.759), 'acc': np.float64(41.667), 'prec': 24.607, 'rec': 65.323, 'f1': 34.422, 'auc': np.float64(0.498), 'mAP': np.float64(0.311)}
    BDC-BDR AUC 0.52 AP 0.25 TN/FP/FN/TP [11 18  3  4]
    Caries AUC 0.45 AP 0.54 TN/FP/FN/TP [ 2 15  2 17]
    Fractured Teeth AUC 0.57 AP 0.17 TN/FP/FN/TP [17 15  3  1]
    Impacted teeth AUC 0.47 AP 0.43 TN/FP/FN/TP [ 5 19  3  9]
    Infection AUC 0.48 AP 0.16 TN/FP/FN/TP [ 5 26  1  4]
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 181MB/s] 


inception_v3 {'val_auc': np.float64(0.748), 'acc': np.float64(70.556), 'prec': 25.0, 'rec': 28.96, 'f1': 26.559, 'auc': np.float64(0.627), 'mAP': np.float64(0.41)}
    BDC-BDR AUC 0.69 AP 0.37 TN/FP/FN/TP [23  6  5  2]
    Caries AUC 0.67 AP 0.70 TN/FP/FN/TP [10  7  8 11]
    Fractured Teeth AUC 0.64 AP 0.23 TN/FP/FN/TP [32  0  4  0]
    Impacted teeth AUC 0.71 AP 0.57 TN/FP/FN/TP [13 11  5  7]
    Infection AUC 0.43 AP 0.18 TN/FP/FN/TP [29  2  5  0]
{'resnet50': {'val_auc': np.float64(0.6388676027105199), 'acc': np.float64(67.77777777777779), 'prec': 26.022727272727273, 'rec': 31.378446115288217, 'f1': 27.697478991596636, 'auc': np.float64(0.5922755448534037), 'mAP': np.float64(0.41528143655069616)}, 'densenet121': {'val_auc': np.float64(0.7738787252952751), 'acc': np.float64(71.66666666666667), 'prec': 24.500000000000004, 'rec': 26.71679197994988, 'f1': 24.761904761904763, 'auc': np.float64(0.564576965106828), 'mAP': np.float64(0.36689467619378036)}, 'efficientnet_b0': {'val_auc': np